# Symbolic Mathematics & Reinforcement Learning for ODEs

This notebook demonstrates using neural networks to solve Symbolic Mathematics tasks (specifically Ordinary Differential Equations) and optimizing them using Policy-Gradient Reinforcement Learning.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


## 1. Environment Setup & Repository Cloning

In [ ]:
%cd /content

!git clone https://github.com/facebookresearch/SymbolicMathematics.git
%cd /content/SymbolicMathematics

/content
Cloning into 'SymbolicMathematics'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 28 (delta 2), reused 0 (delta 0), pack-reused 20 (from 1)
Receiving objects: 100% (28/28), 61.35 KiB | 981.00 KiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/SymbolicMathematics


In [ ]:
!ls

beam_integration.ipynb	CONTRIBUTING.md  main.py    split_data.py
CODE_OF_CONDUCT.md	LICENSE		 README.md  src


In [ ]:
!pip install -q numpy sympy

In [ ]:
# Download the pre-trained model required by beam_integration.ipynb
!wget -q https://dl.fbaipublicfiles.com/SymbolicMathematics/models/fwd_bwd_ibp.pth

## 2. Load Pre-trained Symbolic Integration Model

In [ ]:
!ls -lh fwd_bwd_ibp.pth

-rw-r--r-- 1 root root 1.8G Mar 14  2020 fwd_bwd_ibp.pth


In [ ]:
import os
import numpy as np
import sympy as sp
import torch

from src.utils import AttrDict
from src.envs import build_env
from src.model import build_modules
from src.utils import to_cuda
from src.envs.sympy_utils import simplify

print("Imports successful!")

Imports successful!


In [ ]:
model_path = "./fwd_bwd_ibp.pth"

assert os.path.isfile(model_path), "Model checkpoint not found!"
print("Model found:", model_path)

Model found: ./fwd_bwd_ibp.pth


## 3. Symbolic Integration Inference Demo

In [ ]:
params = AttrDict({

    # Environment parameters
    'env_name': 'char_sp',
    'int_base': 10,
    'balanced': False,
    'positive': True,
    'precision': 10,
    'n_variables': 1,
    'n_coefficients': 0,
    'leaf_probs': '0.75,0,0.25,0',
    'max_len': 512,
    'max_int': 5,
    'max_ops': 15,
    'max_ops_G': 15,
    'clean_prefix_expr': True,
    'rewrite_functions': '',
    'tasks': 'prim_fwd',
    'operators': 'add:10,sub:3,mul:10,div:5,sqrt:4,pow2:4,pow3:2,pow4:1,pow5:1,ln:4,exp:4,sin:4,cos:4,tan:4,asin:1,acos:1,atan:1,sinh:1,cosh:1,tanh:1,asinh:1,acosh:1,atanh:1',

    # Model parameters
    'cpu': False,
    'emb_dim': 1024,
    'n_enc_layers': 6,
    'n_dec_layers': 6,
    'n_heads': 8,
    'dropout': 0,
    'attention_dropout': 0,
    'sinusoidal_embeddings': False,
    'share_inout_emb': True,
    'reload_model': model_path,
})

In [ ]:
env = build_env(params)
x = env.local_dict['x']

modules = build_modules(env, params)

encoder = modules['encoder']
decoder = modules['decoder']

encoder.eval()
decoder.eval()

print("Model loaded successfully!")

Model loaded successfully!


In [ ]:
F_infix = 'x**3 + sin(x)'

In [ ]:
F = sp.S(F_infix, locals=env.local_dict)

f = F.diff(x)

print("F(x) =", F)
print("f(x) = F'(x) =", f)

F(x) = x**3 + sin(x)
f(x) = F'(x) = 3*x**2 + cos(x)


In [ ]:
F_prefix = env.sympy_to_prefix(F)
f_prefix = env.sympy_to_prefix(f)

print("F prefix:")
print(F_prefix)

print("\nf prefix:")
print(f_prefix)

F prefix:
['add', 'pow', 'x', 'INT+', '3', 'sin', 'x']

f prefix:
['add', 'mul', 'INT+', '3', 'pow', 'x', 'INT+', '2', 'cos', 'x']


In [ ]:
x1_prefix = env.clean_prefix(
    ['sub', 'derivative', 'f', 'x', 'x'] + f_prefix
)

x1 = torch.LongTensor(
    [env.eos_index] +
    [env.word2id[w] for w in x1_prefix] +
    [env.eos_index]
).view(-1, 1)

len1 = torch.LongTensor([len(x1)])

x1, len1 = to_cuda(x1, len1)

with torch.no_grad():
    encoded = encoder(
        'fwd',
        x=x1,
        lengths=len1,
        causal=False
    ).transpose(0, 1)

print("Input encoded successfully.")

Input encoded successfully.


In [ ]:
beam_size = 10

with torch.no_grad():
    _, _, beam = decoder.generate_beam(
        encoded,
        len1,
        beam_size=beam_size,
        length_penalty=1.0,
        early_stopping=1,
        max_len=200
    )

assert len(beam) == 1

hypotheses = beam[0].hyp

print("Number of hypotheses:", len(hypotheses))

Number of hypotheses: 10


In [ ]:
print("Input function f:")
print(f)

print("\nReference function F:")
print(F)

print("\nModel predictions:\n")

for score, sent in sorted(hypotheses, key=lambda x: x[0], reverse=True):

    ids = sent[1:].tolist()
    tok = [env.id2word[wid] for wid in ids]

    try:
        hyp = env.prefix_to_infix(tok)
        hyp = env.infix_to_sympy(hyp)

        # Differentiate the predicted primitive.
        # If its derivative equals the input f,
        # the prediction is mathematically valid.
        res = (
            "OK"
            if simplify(hyp.diff(x) - f, seconds=1) == 0
            else "NO"
        )

    except Exception:
        res = "INVALID PREFIX EXPRESSION"
        hyp = tok

    print(f"{score:.5f}  {res}  {hyp}")

Input function f:
3*x**2 + cos(x)

Reference function F:
x**3 + sin(x)

Model predictions:

-0.13501  OK  x*(x**2 + sin(x)/x)
-0.19097  OK  sin(x) + (x**4 + x)/x
-0.19097  OK  x*(x**2 + 1) - x + sin(x)
-0.19173  OK  x**2*(x + sin(x)/x**2)
-0.22458  OK  x*(x**2 - 1) + x + sin(x)
-0.24124  OK  sin(x) + (x**4 + 2*x)/x
-0.24802  OK  sin(x) + (x**4 - x)/x
-0.24986  OK  (x**4 + x*sin(x))/x
-0.26074  OK  x**3 + cos(x)*tan(x)
-0.26413  OK  x**3 + sin(x)


## 4. Pre-trained ODE Models (First and Second Order)

In [ ]:
def test_expression(F_infix, beam_size=10):

    # Parse reference function
    F = sp.S(F_infix, locals=env.local_dict)

    # Compute derivative: this becomes the model input
    f = F.diff(x)

    # Convert to prefix notation
    F_prefix = env.sympy_to_prefix(F)
    f_prefix = env.sympy_to_prefix(f)

    # Encode input
    x1_prefix = env.clean_prefix(
        ['sub', 'derivative', 'f', 'x', 'x'] + f_prefix
    )

    x1 = torch.LongTensor(
        [env.eos_index] +
        [env.word2id[w] for w in x1_prefix] +
        [env.eos_index]
    ).view(-1, 1)

    len1 = torch.LongTensor([len(x1)])
    x1, len1 = to_cuda(x1, len1)

    # Encode
    with torch.no_grad():
        encoded = encoder(
            'fwd',
            x=x1,
            lengths=len1,
            causal=False
        ).transpose(0, 1)

    # Beam search
    with torch.no_grad():
        _, _, beam = decoder.generate_beam(
            encoded,
            len1,
            beam_size=beam_size,
            length_penalty=1.0,
            early_stopping=1,
            max_len=200
        )

    hypotheses = beam[0].hyp

    print("=" * 80)
    print("REFERENCE F(x):")
    print(F)

    print("\nMODEL INPUT f(x):")
    print(f)

    print("\nPREDICTIONS:")
    print("-" * 80)

    for score, sent in sorted(hypotheses, key=lambda x: x[0], reverse=True):

        ids = sent[1:].tolist()
        tok = [env.id2word[wid] for wid in ids]

        try:
            hyp = env.prefix_to_infix(tok)
            hyp = env.infix_to_sympy(hyp)

            valid = simplify(
                hyp.diff(x) - f,
                seconds=1
            ) == 0

            status = "OK" if valid else "NO"

        except Exception:
            hyp = tok
            status = "INVALID"

        print(f"{score: .5f}  {status:>8}  {hyp}")

    print("=" * 80)

In [ ]:
test_expression('x**2 + 2*x + 1')

REFERENCE F(x):
x**2 + 2*x + 1

MODEL INPUT f(x):
2*x + 2

PREDICTIONS:
--------------------------------------------------------------------------------
-0.33256        OK  x*(x + (2*x + 2)/x)
-0.33417        OK  x*(x + (2*x + 1)/x)
-0.39048        OK  x*(x + (x + 1)/x) + x
-0.39391        OK  x*(x + (x + 2)/x) + x
-0.41401        OK  x**2 + 2*x + tan(1)
-0.41921        OK  x**2 + 2*x + sin(1)
-0.43486        OK  x**2 + 2*x + log(2)
-0.43743        OK  x**2 + 2*x + 2*tan(1)
-0.45717        OK  x*(x + cos(1)/x) + 2*x
-0.46167        OK  x*(x + cos(2)/x) + 2*x


In [ ]:
!wget -q -O ode1.pth https://dl.fbaipublicfiles.com/SymbolicMathematics/models/ode1.pth
!wget -q -O ode2.pth https://dl.fbaipublicfiles.com/SymbolicMathematics/models/ode2.pth

!ls -lh ode1.pth ode2.pth

-rw-r--r-- 1 root root 1.8G Mar 14  2020 ode1.pth
-rw-r--r-- 1 root root 1.8G Mar 14  2020 ode2.pth


In [ ]:
ode_params = AttrDict({
    'env_name': 'char_sp',

    # Environment
    'int_base': 10,
    'balanced': False,
    'positive': True,
    'precision': 10,

    # IMPORTANT: keep this 0
    'n_variables': 1,
    'n_coefficients': 0,

    'leaf_probs': '0.75,0,0.25,0',

    'max_len': 512,
    'max_int': 5,
    'max_ops': 15,
    'max_ops_G': 15,

    'clean_prefix_expr': True,
    'rewrite_functions': '',

    'tasks': 'ode1',

    'operators':
        'add:10,sub:3,mul:10,div:5,'
        'sqrt:4,pow2:4,pow3:2,pow4:1,pow5:1,'
        'ln:4,exp:4,sin:4,cos:4,tan:4,'
        'asin:1,acos:1,atan:1,'
        'sinh:1,cosh:1,tanh:1,'
        'asinh:1,acosh:1,atanh:1',

    # Transformer
    'cpu': False,
    'emb_dim': 1024,
    'n_enc_layers': 6,
    'n_dec_layers': 6,
    'n_heads': 8,
    'dropout': 0,
    'attention_dropout': 0,
    'sinusoidal_embeddings': False,
    'share_inout_emb': True,

    # ODE1 checkpoint
    'reload_model': './ode1.pth',
})

In [ ]:
ode_env = build_env(ode_params)

ode_modules = build_modules(ode_env, ode_params)

ode_encoder = ode_modules['encoder']
ode_decoder = ode_modules['decoder']

ode_encoder.eval()
ode_decoder.eval()

print("ODE1 model loaded successfully!")

ODE1 model loaded successfully!


In [ ]:
def test_ode(equation, order=1, beam_size=10):

    assert order in [1, 2]

    # ---------------------------------------------------------
    # Load correct model
    # ---------------------------------------------------------
    model_path = './ode1.pth' if order == 1 else './ode2.pth'
    task = 'ode1' if order == 1 else 'ode2'

    params = AttrDict({
        # Environment
        'env_name': 'char_sp',
        'int_base': 10,
        'balanced': False,
        'positive': True,
        'precision': 10,
        'n_variables': 1,
        'n_coefficients': 0,
        'leaf_probs': '0.75,0,0.25,0',
        'max_len': 512,
        'max_int': 5,
        'max_ops': 15,
        'max_ops_G': 15,
        'clean_prefix_expr': True,
        'rewrite_functions': '',
        'tasks': task,

        'operators':
            'add:10,sub:3,mul:10,div:5,'
            'sqrt:4,pow2:4,pow3:2,pow4:1,pow5:1,'
            'ln:4,exp:4,sin:4,cos:4,tan:4,'
            'asin:1,acos:1,atan:1,'
            'sinh:1,cosh:1,tanh:1,'
            'asinh:1,acosh:1,atanh:1',

        # Model
        'cpu': False,
        'emb_dim': 1024,
        'n_enc_layers': 6,
        'n_dec_layers': 6,
        'n_heads': 8,
        'dropout': 0,
        'attention_dropout': 0,
        'sinusoidal_embeddings': False,
        'share_inout_emb': True,
        'reload_model': model_path,
    })

    # ---------------------------------------------------------
    # Build environment and model
    # ---------------------------------------------------------
    env = build_env(params)

    x = env.local_dict['x']
    f = env.local_dict['f']

    modules = build_modules(env, params)

    encoder = modules['encoder']
    decoder = modules['decoder']

    encoder.eval()
    decoder.eval()

    # ---------------------------------------------------------
    # Display equation
    # ---------------------------------------------------------
    print("=" * 80)
    print(f"ODE{order}")
    print("Input equation:")
    print(equation)

    # ---------------------------------------------------------
    # Convert equation to prefix notation
    # ---------------------------------------------------------
    eq_prefix = env.sympy_to_prefix(equation)

    # Convert:
    # f(x)       -> Y
    # f'(x)      -> Y'
    # f''(x)     -> Y''
    x1_prefix = env.clean_prefix(eq_prefix)

    print("\nPrefix representation:")
    print(eq_prefix)

    print("\nClean prefix representation:")
    print(x1_prefix)

    # ---------------------------------------------------------
    # Encode input
    # ---------------------------------------------------------
    x1 = torch.LongTensor(
        [env.eos_index] +
        [env.word2id[w] for w in x1_prefix] +
        [env.eos_index]
    ).view(-1, 1)

    len1 = torch.LongTensor([len(x1)])

    x1, len1 = to_cuda(x1, len1)

    # ---------------------------------------------------------
    # Encoder
    # ---------------------------------------------------------
    with torch.no_grad():
        encoded = encoder(
            'fwd',
            x=x1,
            lengths=len1,
            causal=False
        ).transpose(0, 1)

    # ---------------------------------------------------------
    # Beam search
    # ---------------------------------------------------------
    with torch.no_grad():
        _, _, beam = decoder.generate_beam(
            encoded,
            len1,
            beam_size=beam_size,
            length_penalty=1.0,
            early_stopping=1,
            max_len=params.max_len
        )

    hypotheses = beam[0].hyp

    # ---------------------------------------------------------
    # Display predictions
    # ---------------------------------------------------------
    print("\nPredictions:")
    print("-" * 80)

    for score, sequence in sorted(
        hypotheses,
        reverse=True
    ):

        ids = sequence[1:].tolist()

        hyp_prefix = [
            env.id2word[word_id]
            for word_id in ids
        ]

        try:
            hyp_infix = env.prefix_to_infix(hyp_prefix)
            hyp_sympy = env.infix_to_sympy(hyp_infix)

            # Substitute predicted y(x) into ODE
            validation = (
                simplify(
                    equation.subs(f(x), hyp_sympy).doit(),
                    seconds=1
                ) == 0
            )

            status = "OK" if validation else "NO"

            print(
                f"{score: .5f}   "
                f"{status:>3}   "
                f"{hyp_sympy}"
            )

        except Exception as e:
            print(
                f"{score: .5f}   "
                f"INVALID   "
                f"{hyp_prefix}"
            )

    print("=" * 80)

In [ ]:
x = ode_env.local_dict['x']
f = ode_env.local_dict['f']

In [ ]:
ode1 = sp.diff(f(x), x) - f(x)

test_ode(
    ode1,
    order=1
)

ODE1
Input equation:
-f(x) + Derivative(f(x), x)

Prefix representation:
['add', 'mul', 'INT-', '1', 'f', 'x', 'derivative', 'f', 'x', 'x']

Clean prefix representation:
['add', 'mul', 'INT-', '1', 'Y', "Y'"]

Predictions:
--------------------------------------------------------------------------------
-0.44210    OK   (a8*x + x)*exp(x + 1)/x
-0.44287    OK   (a8*x + x)*exp(x + 5)/x
-0.44395    OK   (a8*x + x)*exp(x + 2)/x
-0.44656    OK   (a8*x - x)*exp(x)/x
-0.44734    OK   (a8*x + x)*exp(x + 4)/x
-0.44989    OK   (a8*x + x)*exp(x + 3)/x
-0.46156    OK   (a8*x + 2*x)*exp(x)/x
-0.50374    OK   a8*exp(x + pi/4)
-0.55816    OK   a8*exp(x + pi/2)
-0.57683    OK   (a8*x + x)*exp(x)/x


In [ ]:
ode2 = sp.diff(f(x), x, 2) + f(x)

test_ode(
    ode2,
    order=2
)

ODE2
Input equation:
f(x) + Derivative(f(x), (x, 2))

Prefix representation:
['add', 'derivative', 'derivative', 'f', 'x', 'x', 'x', 'f', 'x']

Clean prefix representation:
['add', "Y''", 'Y']

Predictions:
--------------------------------------------------------------------------------
-0.38320    OK   a9*cos(x) - sin(a8 - x)
-0.38663    OK   a9*(sin(x) - sin(a8 - x))
-0.39379    OK   a9*(-sin(a8 - x) + cos(x))
-0.40663    OK   a9*sin(x) - sin(a8 - x)
-0.41302    OK   a8*(-sin(a9 - x) + cos(x))
-0.41315    OK   a8*(sin(x) - sin(a9 - x))
-0.46157    OK   a9*(sin(x) - sin(a8 + x))
-0.46174    OK   a9*cos(x) - sin(a8 + x)
-0.46377    OK   a9*(-sin(a8 + x) + cos(x))
-0.47814    OK   a9*(-sin(x) + sin(a8 + x))


In [ ]:
ode2 = sp.diff(f(x), x, 2) - 4*f(x)

test_ode(
    ode2,
    order=2
)

ODE2
Input equation:
-4*f(x) + Derivative(f(x), (x, 2))

Prefix representation:
['add', 'mul', 'INT-', '4', 'f', 'x', 'derivative', 'derivative', 'f', 'x', 'x', 'x']

Clean prefix representation:
['add', 'mul', 'INT-', '4', 'Y', "Y''"]

Predictions:
--------------------------------------------------------------------------------
-0.24704    OK   x*sinh(a9 + 2*x)/(a8*x + x)
-0.25362    OK   x*cosh(a9 + 2*x)/(a8*x + x)
-0.25438    OK   x*sinh(a8 + 2*x)/(a9*x + x)
-0.26763    OK   x*cosh(a8 + 2*x)/(a9*x + x)
-0.27177    OK   a9*sinh(a8 + x + (x**2 + x)/x)
-0.27216    OK   (a9*x + x)*sinh(a8 + 2*x)/x
-0.28688    OK   (a9*x + x)*cosh(a8 + 2*x)/x
-0.29137    OK   (a8*x + x)*cosh(a9 + 2*x)/x
-0.29376    OK   (a8*x + x)*sinh(a9 + 2*x)/x
-0.40835    OK   a8*cosh(a9 - 2*x)


## 5. Analyzing the Lane-Emden Astrophysical Equation

In [ ]:
lane_emden = {}

for n in [0, 1, 2]:
    equation = (
        sp.diff(f(x), x, 2)
        + sp.Rational(2, 1) / x * sp.diff(f(x), x)
        + f(x)**n
    )

    lane_emden[n] = equation

    print(f"\nn = {n}")
    print(equation)


n = 0
Derivative(f(x), (x, 2)) + 1 + 2*Derivative(f(x), x)/x

n = 1
f(x) + Derivative(f(x), (x, 2)) + 2*Derivative(f(x), x)/x

n = 2
f(x)**2 + Derivative(f(x), (x, 2)) + 2*Derivative(f(x), x)/x


In [ ]:
test_ode(
    lane_emden[0],
    order=2,
    beam_size=10
)

ODE2
Input equation:
Derivative(f(x), (x, 2)) + 1 + 2*Derivative(f(x), x)/x

Prefix representation:
['add', 'INT+', '1', 'add', 'mul', 'INT+', '2', 'mul', 'pow', 'x', 'INT-', '1', 'derivative', 'f', 'x', 'x', 'derivative', 'derivative', 'f', 'x', 'x', 'x']

Clean prefix representation:
['add', 'INT+', '1', 'add', 'mul', 'INT+', '2', 'mul', 'pow', 'x', 'INT-', '1', "Y'", "Y''"]

Predictions:
--------------------------------------------------------------------------------
-0.21101    NO   a9/x - x + log(x/(a8*x + x))
-0.25756    NO   a9/x - x + log(1 + (a8 + x)/x)
-0.26053    NO   a9/x - x + log(x/(a8*x + x)**2)
-0.28959    NO   a9*(a8*x**2 + x)/x - x + log(x)
-0.29384    NO   a9/x + log(x/(a8*x + x)**2)
-0.29444    NO   a9/x - x + log((a8*x + x)/x)
-0.31064    NO   a9/x - x + log(a8/x)
-0.31104    NO   a9/x - x + log(a8*sqrt(1/x)/x)
-0.31290    NO   a9/x + log(x/(a8*x + x))
-0.33036    NO   a9/x - x + log(a8*x + x)


In [ ]:
theta_0 = 1 - x**2 / 6

print("Reference Lane-Emden solution:")
print(theta_0)

Reference Lane-Emden solution:
1 - x**2/6


In [ ]:
residual_0 = sp.simplify(
    lane_emden[0].subs(f(x), theta_0).doit()
)

print("Residual:")
print(residual_0)

Residual:
0


In [ ]:
test_ode(
    lane_emden[1],
    order=2,
    beam_size=10
)

ODE2
Input equation:
f(x) + Derivative(f(x), (x, 2)) + 2*Derivative(f(x), x)/x

Prefix representation:
['add', 'mul', 'INT+', '2', 'mul', 'pow', 'x', 'INT-', '1', 'derivative', 'f', 'x', 'x', 'add', 'derivative', 'derivative', 'f', 'x', 'x', 'x', 'f', 'x']

Clean prefix representation:
['add', 'mul', 'INT+', '2', 'mul', 'pow', 'x', 'INT-', '1', "Y'", 'add', "Y''", 'Y']

Predictions:
--------------------------------------------------------------------------------
-0.18177    NO   a9*(a8 + x)*exp(-x)/(x*sqrt(1 - a9**2))
-0.19535    NO   a9*(x*(a8 + x) + x)*exp(-x)/(x*sqrt(1 - a9**2))
-0.21224    NO   a9*(a8*x**2 + x)*exp(-x)/(x*sqrt(a9**2 + 1))
-0.21881    NO   a9*(a8 - x)*exp(-x)/(x*sqrt(1 - a9**2))
-0.21939    NO   a9*(a8 + x)*exp(-x)/(x*sqrt(a9**2 + 1))
-0.22047    NO   a9*(a8 + 2*x)*exp(-x)/(x*sqrt(1 - a9**2))
-0.23457    NO   x*(a8 + x)*exp(-x)/(a9*x + x)
-0.24281    NO   a9*(a8*x**2 + x)*exp(-x)/(x*(1 - a9))
-0.24612    NO   a9*x*(a8 + x)*exp(-x)/(x*cos(4) + x)
-0.27057    NO   x*(

In [ ]:
theta_1 = sp.sin(x) / x

print("Reference Lane-Emden solution:")
print(theta_1)

Reference Lane-Emden solution:
sin(x)/x


In [ ]:
residual_1 = sp.simplify(
    lane_emden[1].subs(f(x), theta_1).doit()
)

print("Residual:")
print(residual_1)

Residual:
0


In [ ]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to: {seed}")

set_seed(42)

Global seed set to: 42


In [ ]:
test_ode(
    lane_emden[2],
    order=2,
    beam_size=10
)

ODE2
Input equation:
f(x)**2 + Derivative(f(x), (x, 2)) + 2*Derivative(f(x), x)/x

Prefix representation:
['add', 'pow', 'f', 'x', 'INT+', '2', 'add', 'mul', 'INT+', '2', 'mul', 'pow', 'x', 'INT-', '1', 'derivative', 'f', 'x', 'x', 'derivative', 'derivative', 'f', 'x', 'x', 'x']

Clean prefix representation:
['add', 'pow', 'Y', 'INT+', '2', 'add', 'mul', 'INT+', '2', 'mul', 'pow', 'x', 'INT-', '1', "Y'", "Y''"]

Predictions:
--------------------------------------------------------------------------------
-0.22310    NO   x*(a8 + x)*exp(-x)/(a9*x + x)
-0.26290    NO   (a8*x**2 + x)*exp(-x)/(a9*x + x)
-0.26795    NO   x*(a8 - x)*exp(-x)/(a9*x + x)
-0.26988    NO   x*(a8 + 2*x)*exp(-x)/(a9*x + x)
-0.27330    NO   x**2*(a8 + x)*exp(-x)/(a9*x + x)**2
-0.28103    NO   x*exp(-x)*log(a8*exp(x))/(a9*x + x)
-0.30312    NO   x*exp(-x)*log(a8*x)/(a9*x + x)
-0.30524    NO   x*exp(-x)*log(a8/x)/(a9*x + x)
-0.34624    NO   x*exp(-x)*log(a8*x**2)/(a9*x + x)
-0.36796    NO   x*exp(-x)*log(a8*x**3)/(a9*

## 6. Configuring the ODE2 Solver Environment

In [ ]:
from src.utils import AttrDict

ode2_params = AttrDict({
    # ============================================================
    # Environment
    # ============================================================
    'env_name': 'char_sp',

    'int_base': 10,
    'balanced': False,
    'positive': True,
    'precision': 10,

    'n_variables': 1,

    # IMPORTANT:
    # Must remain 0 with leaf_probs[1] == 0
    'n_coefficients': 0,

    'leaf_probs': '0.75,0,0.25,0',

    'max_ops': 15,
    'max_ops_G': 15,
    'max_int': 5,
    'max_len': 512,

    'clean_prefix_expr': True,
    'rewrite_functions': '',

    # ============================================================
    # Task
    # ============================================================
    'tasks': 'ode2',

    # ============================================================
    # Operators
    # ============================================================
    'operators':
        'add:10,sub:3,mul:10,div:5,'
        'sqrt:4,pow2:4,pow3:2,pow4:1,pow5:1,'
        'ln:4,exp:4,sin:4,cos:4,tan:4,'
        'asin:1,acos:1,atan:1,'
        'sinh:1,cosh:1,tanh:1,'
        'asinh:1,acosh:1,atanh:1',

    # ============================================================
    # Transformer
    # ============================================================
    'cpu': False,

    'emb_dim': 1024,

    'n_enc_layers': 6,
    'n_dec_layers': 6,
    'n_heads': 8,

    'dropout': 0,
    'attention_dropout': 0,

    'sinusoidal_embeddings': False,
    'share_inout_emb': True,

    # ============================================================
    # Pretrained ODE2 model
    # ============================================================
    'reload_model': './ode2.pth',
})

In [ ]:
print("env_name     :", ode2_params.env_name)
print("tasks        :", ode2_params.tasks)
print("max_int      :", ode2_params.max_int)
print("max_ops      :", ode2_params.max_ops)
print("max_ops_G    :", ode2_params.max_ops_G)
print("max_len      :", ode2_params.max_len)
print("n_variables  :", ode2_params.n_variables)
print("n_coefficients:", ode2_params.n_coefficients)
print("checkpoint   :", ode2_params.reload_model)

env_name     : char_sp
tasks        : ode2
max_int      : 5
max_ops      : 15
max_ops_G    : 15
max_len      : 512
n_variables  : 1
n_coefficients: 0
checkpoint   : ./ode2.pth


In [ ]:
ode2_env = build_env(ode2_params)

print("Environment built successfully!")

Environment built successfully!


In [ ]:
ode2_modules = build_modules(
    ode2_env,
    ode2_params
)

encoder = ode2_modules['encoder']
decoder = ode2_modules['decoder']

encoder.eval()
decoder.eval()

print("ODE2 model loaded successfully!")

ODE2 model loaded successfully!


In [ ]:
device = next(decoder.parameters()).device

x = ode2_env.local_dict['x']
f = ode2_env.local_dict['f']

print("Device:", device)
print("ODE2 model ready.")

Device: cuda:0
ODE2 model ready.


In [ ]:
def lane_emden_equation(n):
    return (
        sp.diff(f(x), x, 2)
        + sp.Rational(2, 1) / x * sp.diff(f(x), x)
        + f(x)**n
    )


lane_eqs = {
    n: lane_emden_equation(n)
    for n in [0, 1, 2]
}

for n, eq in lane_eqs.items():
    print(f"\nn = {n}")
    print(eq)


n = 0
Derivative(f(x), (x, 2)) + 1 + 2*Derivative(f(x), x)/x

n = 1
f(x) + Derivative(f(x), (x, 2)) + 2*Derivative(f(x), x)/x

n = 2
f(x)**2 + Derivative(f(x), (x, 2)) + 2*Derivative(f(x), x)/x


In [ ]:
exact_solutions = {
    0: 1 - x**2 / 6,
    1: sp.sin(x) / x,
}

print("n=0:", exact_solutions[0])
print("n=1:", exact_solutions[1])

n=0: 1 - x**2/6
n=1: sin(x)/x


## 7. Defining RL Rewards, Objectives & Metrics

In [ ]:
TRAIN_X = np.linspace(1.0, 5.0, 20)

print(TRAIN_X)

[1.         1.21052632 1.42105263 1.63157895 1.84210526 2.05263158
 2.26315789 2.47368421 2.68421053 2.89473684 3.10526316 3.31578947
 3.52631579 3.73684211 3.94736842 4.15789474 4.36842105 4.57894737
 4.78947368 5.        ]


In [ ]:
def hypothesis_to_sympy(env, hypothesis):
    """
    Convert one beam hypothesis into a SymPy expression.
    """

    # hypothesis contains initial EOS/BOS token
    token_ids = hypothesis[1:].tolist()

    tokens = [
        env.id2word[int(idx)]
        for idx in token_ids
    ]

    # Y -> f x
    # Y' -> derivative f x
    # Y'' -> derivative f x x
    tokens = env.unclean_prefix(tokens)

    infix = env.prefix_to_infix(tokens)

    expr = sp.S(
        infix,
        locals=env.local_dict
    )

    return expr

In [ ]:
def instantiate_lane_emden_solution(expr, env):
    """
    Instantiate the symbolic integration constants for the
    regular Lane-Emden solution.

    Standard choice:
        a8 = 1
        a9 = 0
    """

    substitutions = {
        coeff: sp.Integer(0)
        for coeff in env.coefficients.values()
    }

    substitutions[env.local_dict['a8']] = sp.Integer(1)
    substitutions[env.local_dict['a9']] = sp.Integer(0)

    return sp.simplify(
        expr.subs(substitutions)
    )

In [ ]:
def numerical_mse(candidate, reference, x_values):
    """
    Mean squared error between two symbolic functions
    evaluated on a numerical grid.
    """

    try:
        candidate_fn = sp.lambdify(
            x,
            candidate,
            modules='numpy'
        )

        reference_fn = sp.lambdify(
            x,
            reference,
            modules='numpy'
        )

        pred = np.asarray(
            candidate_fn(x_values),
            dtype=np.complex128
        )

        target = np.asarray(
            reference_fn(x_values),
            dtype=np.complex128
        )

        # Reject complex-valued or non-finite candidates
        if not np.all(np.isfinite(pred)):
            return np.inf

        if np.max(np.abs(pred.imag)) > 1e-8:
            return np.inf

        pred = pred.real
        target = target.real

        return float(
            np.mean(
                (pred - target) ** 2
            )
        )

    except Exception:
        return np.inf

In [ ]:
EPS = 1e-12

def loss_to_reward(loss):
    if not np.isfinite(loss):
        return -12.0

    return float(
        -np.log10(loss + EPS)
    )

In [ ]:
print("L = 1e-3  -> reward =", loss_to_reward(1e-3))
print("L = 1e-6  -> reward =", loss_to_reward(1e-6))

L = 1e-3  -> reward = 2.9999999995657056
L = 1e-6  -> reward = 5.9999995657057354


In [ ]:
def encode_equation(equation):
    """
    Convert a SymPy ODE into the model's input representation
    and encode it using the pretrained encoder.
    """

    prefix = ode2_env.sympy_to_prefix(equation)
    prefix = ode2_env.clean_prefix(prefix)

    ids = (
        [ode2_env.eos_index]
        + [ode2_env.word2id[w] for w in prefix]
        + [ode2_env.eos_index]
    )

    x1 = torch.tensor(
        ids,
        dtype=torch.long,
        device=device
    ).view(-1, 1)

    len1 = torch.tensor(
        [len(ids)],
        dtype=torch.long,
        device=device
    )

    encoder.eval()

    with torch.no_grad():
        encoded = encoder(
            'fwd',
            x=x1,
            lengths=len1,
            causal=False
        ).transpose(0, 1)

    return encoded, len1

In [ ]:
encoded_cache = {}

for n in [0, 1, 2]:
    encoded_cache[n] = encode_equation(
        lane_eqs[n]
    )

print("Lane–Emden equations encoded.")

Lane–Emden equations encoded.


In [ ]:
def generate_lane_beam(n, beam_size=10):
    """
    Generate beam_size candidate solutions.
    """

    src_enc, src_len = encoded_cache[n]

    decoder.eval()

    with torch.no_grad():

        _, _, generations = decoder.generate_beam(
            src_enc,
            src_len,
            beam_size=beam_size,
            length_penalty=1.0,
            early_stopping=1,
            max_len=200
        )

    return sorted(
        generations[0].hyp,
        key=lambda z: z[0],
        reverse=True
    )

## 8. Baseline Model Performance Evaluation

In [ ]:
def evaluate_lane_beam(n, x_values=TRAIN_X, verbose=True):

    hypotheses = generate_lane_beam(
        n,
        beam_size=10
    )

    reference = exact_solutions[n]

    results = []

    for rank, (beam_score, hypothesis) in enumerate(
        hypotheses,
        start=1
    ):

        try:
            raw_expr = hypothesis_to_sympy(
                ode2_env,
                hypothesis
            )

            instantiated_expr = instantiate_lane_emden_solution(
                raw_expr,
                ode2_env
            )

            mse = numerical_mse(
                instantiated_expr,
                reference,
                x_values
            )

            reward = loss_to_reward(mse)

        except Exception as e:

            raw_expr = None
            instantiated_expr = None
            mse = np.inf
            reward = -12.0

        result = {
            'rank': rank,
            'beam_score': float(beam_score),
            'hypothesis': hypothesis,
            'raw_expr': raw_expr,
            'expr': instantiated_expr,
            'mse': mse,
            'reward': reward,
        }

        results.append(result)

    if verbose:

        print("=" * 100)
        print(f"Lane–Emden n={n}")
        print("=" * 100)

        for r in results:

            print(
                f"{r['rank']:2d} | "
                f"beam={r['beam_score']:8.4f} | "
                f"MSE={r['mse']:12.4e} | "
                f"reward={r['reward']:8.3f}"
            )

            print(
                f"     {r['expr']}"
            )

    return results

In [ ]:
baseline_n0 = evaluate_lane_beam(0)
baseline_n1 = evaluate_lane_beam(1)

Lane–Emden n=0
 1 | beam= -0.2110 | MSE=  8.7358e+00 | reward=  -0.941
     -x - log(2)
 2 | beam= -0.2576 | MSE=  1.9654e+00 | reward=  -0.293
     -x + log(2 + 1/x)
 3 | beam= -0.2605 | MSE=  2.1849e+01 | reward=  -1.339
     -x + log(1/(4*x))
 4 | beam= -0.2896 | MSE=  1.7957e+00 | reward=  -0.254
     -x + log(x)
 5 | beam= -0.2938 | MSE=  3.3677e+00 | reward=  -0.527
     log(1/(4*x))
 6 | beam= -0.2944 | MSE=  2.4854e+00 | reward=  -0.395
     -x + log(2)
 7 | beam= -0.3106 | MSE=  1.0902e+01 | reward=  -1.038
     -x + log(1/x)
 8 | beam= -0.3110 | MSE=  1.4703e+01 | reward=  -1.167
     -x + log(sqrt(1/x)/x)
 9 | beam= -0.3129 | MSE=  1.5243e+00 | reward=  -0.183
     -log(2)
10 | beam= -0.3304 | MSE=  5.3820e-01 | reward=   0.269
     -x + log(2*x)
Lane–Emden n=1
 1 | beam= -0.1818 | MSE=  1.5610e-01 | reward=   0.807
     0
 2 | beam= -0.1954 | MSE=  1.5610e-01 | reward=   0.807
     0
 3 | beam= -0.2122 | MSE=  1.5610e-01 | reward=   0.807
     0
 4 | beam= -0.2188 | MSE=  1

## 9. Policy Gradient (RL) Fine-Tuning

In [ ]:
import torch.nn.functional as F

def candidate_logprobs(
    hypotheses,
    src_enc,
    src_len
):
    """
    Calculate differentiable average log P(candidate | input)
    for all beam candidates.

    hypotheses:
        list of (beam_score, hypothesis_tensor)
    """

    sequences = []

    # --------------------------------------------------------
    # Extract the actual token tensor from each
    # (score, tensor) hypothesis tuple.
    # --------------------------------------------------------

    for beam_score, hypothesis in hypotheses:

        # Beam hypothesis contains the initial <EOS>/<BOS>.
        # batch_sequences() will add BOS/EOS itself, so remove
        # the first token here.
        target_tokens = hypothesis[1:].long().cpu()

        sequences.append(target_tokens)

    # --------------------------------------------------------
    # Build a padded batch.
    #
    # batch_sequences() adds:
    #
    #   <EOS> candidate <EOS>
    #
    # exactly as the repository's training data does.
    # --------------------------------------------------------

    x2, len2 = ode2_env.batch_sequences(
        sequences
    )

    x2 = x2.to(device)
    len2 = len2.to(device)

    batch_size = len(sequences)

    # --------------------------------------------------------
    # Same encoded input for every candidate.
    # --------------------------------------------------------

    src_enc_batch = src_enc.expand(
        batch_size,
        -1,
        -1
    )

    src_len_batch = src_len.expand(
        batch_size
    )

    # --------------------------------------------------------
    # Teacher-forced decoder pass.
    # --------------------------------------------------------

    decoded = decoder(
        'fwd',
        x=x2,
        lengths=len2,
        causal=True,
        src_enc=src_enc_batch,
        src_len=src_len_batch
    )

    # decoded[t] predicts x2[t+1]
    logits = decoder.proj(
        decoded[:-1]
    )

    log_probs = F.log_softmax(
        logits,
        dim=-1
    )

    targets = x2[1:]

    token_log_probs = log_probs.gather(
        -1,
        targets.unsqueeze(-1)
    ).squeeze(-1)

    # --------------------------------------------------------
    # Ignore padding.
    # --------------------------------------------------------

    time = torch.arange(
        token_log_probs.size(0),
        device=device
    )

    mask = (
        time[:, None]
        < (len2 - 1)[None, :]
    )

    token_log_probs = (
        token_log_probs * mask
    )

    # Average token log probability for each candidate.
    sequence_log_probs = (
        token_log_probs.sum(dim=0)
        / (len2 - 1).float()
    )

    return sequence_log_probs

In [ ]:
test_hypotheses = generate_lane_beam(
    0,
    beam_size=10
)

print("Number of hypotheses:", len(test_hypotheses))

for i, (score, hyp) in enumerate(test_hypotheses):
    print(
        i + 1,
        "score =", score,
        "type =", type(hyp),
        "shape =", hyp.shape
    )

Number of hypotheses: 10
1 score = -0.2110094428062439 type = <class 'torch.Tensor'> shape = torch.Size([24])
2 score = -0.25756307601928713 type = <class 'torch.Tensor'> shape = torch.Size([25])
3 score = -0.2605295379956563 type = <class 'torch.Tensor'> shape = torch.Size([24])
4 score = -0.28959476947784424 type = <class 'torch.Tensor'> shape = torch.Size([24])
5 score = -0.2938447249563117 type = <class 'torch.Tensor'> shape = torch.Size([19])
6 score = -0.29443732897440594 type = <class 'torch.Tensor'> shape = torch.Size([24])
7 score = -0.3106433391571045 type = <class 'torch.Tensor'> shape = torch.Size([20])
8 score = -0.31104036477895886 type = <class 'torch.Tensor'> shape = torch.Size([26])
9 score = -0.3128960509049265 type = <class 'torch.Tensor'> shape = torch.Size([19])
10 score = -0.33036367516768606 type = <class 'torch.Tensor'> shape = torch.Size([19])


In [ ]:
src_enc, src_len = encoded_cache[0]

log_probs = candidate_logprobs(
    test_hypotheses,
    src_enc,
    src_len
)

print("Shape:", log_probs.shape)
print("Log probabilities:")
print(log_probs)

Shape: torch.Size([10])
Log probabilities:
tensor([-0.2110, -0.2576, -0.2605, -0.2896, -0.2938, -0.2944, -0.3106, -0.3110,
        -0.3129, -0.3304], device='cuda:0', grad_fn=<DivBackward0>)


In [ ]:
for parameter in encoder.parameters():
    parameter.requires_grad = False

for parameter in decoder.parameters():
    parameter.requires_grad = True

encoder.eval()

TransformerModel(
  (position_embeddings): Embedding(4096, 1024)
  (embeddings): Embedding(91, 1024, padding_idx=1)
  (layer_norm_emb): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
  (attentions): ModuleList(
    (0-5): 6 x MultiHeadAttention(
      (q_lin): Linear(in_features=1024, out_features=1024, bias=True)
      (k_lin): Linear(in_features=1024, out_features=1024, bias=True)
      (v_lin): Linear(in_features=1024, out_features=1024, bias=True)
      (out_lin): Linear(in_features=1024, out_features=1024, bias=True)
    )
  )
  (layer_norm1): ModuleList(
    (0-5): 6 x LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
  )
  (ffns): ModuleList(
    (0-5): 6 x TransformerFFN(
      (lin1): Linear(in_features=1024, out_features=4096, bias=True)
      (lin2): Linear(in_features=4096, out_features=1024, bias=True)
    )
  )
  (layer_norm2): ModuleList(
    (0-5): 6 x LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
  )
)

In [ ]:
optimizer = torch.optim.AdamW(
    decoder.parameters(),
    lr=1e-6,
    weight_decay=1e-4
)

print("RL optimizer ready.")

RL optimizer ready.


In [ ]:
from torch.nn.utils import clip_grad_norm_

def rl_update_step(
    train_ns=(0, 1),
    beam_size=10
):

    # --------------------------------------------------------
    # 1. Generate candidates in evaluation mode
    # --------------------------------------------------------
    decoder.eval()

    all_data = {}

    for n in train_ns:

        hypotheses = generate_lane_beam(
            n,
            beam_size=beam_size
        )

        reference = exact_solutions[n]

        rewards = []

        for _, hypothesis in hypotheses:

            try:
                raw_expr = hypothesis_to_sympy(
                    ode2_env,
                    hypothesis
                )

                expr = instantiate_lane_emden_solution(
                    raw_expr,
                    ode2_env
                )

                mse = numerical_mse(
                    expr,
                    reference,
                    TRAIN_X
                )

                reward = loss_to_reward(mse)

            except Exception:
                reward = -12.0

            rewards.append(reward)

        rewards = torch.tensor(
            rewards,
            dtype=torch.float32,
            device=device
        )

        # ----------------------------------------------------
        # Advantage normalization
        # ----------------------------------------------------
        advantages = (
            rewards - rewards.mean()
        ) / (
            rewards.std() + 1e-6
        )

        advantages = torch.clamp(
            advantages,
            -3.0,
            3.0
        )

        all_data[n] = {
            "hypotheses": hypotheses,
            "rewards": rewards,
            "advantages": advantages,
        }

    # --------------------------------------------------------
    # 2. Compute differentiable candidate log-probabilities
    #
    # IMPORTANT:
    # We do NOT use torch.no_grad() here.
    # eval() is fine; gradients are still enabled.
    # --------------------------------------------------------

    total_loss = 0.0

    for n in train_ns:

        hypotheses = all_data[n]["hypotheses"]
        advantages = all_data[n]["advantages"]

        src_enc, src_len = encoded_cache[n]

        log_probs = candidate_logprobs(
            hypotheses,
            src_enc,
            src_len
        )

        # ----------------------------------------------------
        # Policy-gradient surrogate
        # ----------------------------------------------------

        loss_n = -(
            advantages.detach()
            * log_probs
        ).mean()

        total_loss = total_loss + loss_n

    total_loss /= len(train_ns)

    # --------------------------------------------------------
    # 3. Backpropagation
    # --------------------------------------------------------

    optimizer.zero_grad(set_to_none=True)

    total_loss.backward()

    grad_norm = clip_grad_norm_(
        decoder.parameters(),
        max_norm=1.0
    )

    # --------------------------------------------------------
    # 4. Update weights
    # --------------------------------------------------------

    optimizer.step()

    decoder.eval()

    all_rewards = torch.cat([
        all_data[n]["rewards"]
        for n in train_ns
    ])

    return {
        "loss": total_loss.item(),
        "avg_reward": all_rewards.mean().item(),
        "grad_norm": (
            grad_norm.item()
            if torch.is_tensor(grad_norm)
            else float(grad_norm)
        ),
        "data": all_data,
    }

In [ ]:
import matplotlib.pyplot as plt

# Configure training steps
NUM_STEPS = 500
steps = []
losses = []
rewards = []

print("Starting RL training loop and collecting metrics...")
for step in range(1, NUM_STEPS + 1):
    metrics = rl_update_step(train_ns=(0, 1), beam_size=10)

    steps.append(step)
    losses.append(metrics['loss'])
    rewards.append(metrics['avg_reward'])

    if step == 1 or step % 10 == 0:
        print(
            f"Step {step:3d} | "
            f"RL loss = {metrics['loss']:.5f} | "
            f"avg reward = {metrics['avg_reward']:.4f}"
        )

print("\nTraining complete! Generating performance plots...")

# Create the combined metrics plot
fig, ax1 = plt.subplots(figsize=(10, 5))

# Plot RL Loss
color = 'tab:red'
ax1.set_xlabel('Training Steps')
ax1.set_ylabel('RL Loss', color=color)
ax1.plot(steps, losses, color=color, label='RL Loss', linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, linestyle='--', alpha=0.5)

# Plot Average Reward
ax2 = ax1.twinx()
color = 'tab:blue'
ax2.set_ylabel('Average Reward', color=color)
ax2.plot(steps, rewards, color=color, label='Average Reward', linewidth=2)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Reinforcement Learning Training Progress (Loss & Reward)')
fig.tight_layout()
plt.show()

Starting RL training loop and collecting metrics...
Step   1 | RL loss = -0.07096 | avg reward = 0.2551
Step  10 | RL loss = -0.04215 | avg reward = 0.8293
Step  20 | RL loss = -0.03227 | avg reward = 0.8007
Step  30 | RL loss = -0.02647 | avg reward = 0.9291
Step  40 | RL loss = -0.02688 | avg reward = 0.8985
Step  50 | RL loss = -0.02146 | avg reward = 0.9439


<lambdifygenerated-10549>:2: RuntimeWarning: invalid value encountered in log
  return -x + log(-1/2*x)
<lambdifygenerated-10591>:2: RuntimeWarning: invalid value encountered in log
  return -x + log(-1/2*x)


Step  60 | RL loss = -0.02201 | avg reward = 0.3023
Step  70 | RL loss = -0.05029 | avg reward = 0.8292


<lambdifygenerated-11043>:2: RuntimeWarning: divide by zero encountered in log
  return -x + log(2*x*sinh(log(x)))


Step  80 | RL loss = -0.04446 | avg reward = 0.8838


...

Step 350 | RL loss = -0.03942 | avg reward = 1.1073
Step 360 | RL loss = -0.02455 | avg reward = 1.0309
Step 370 | RL loss = -0.02214 | avg reward = 1.1396
Step 380 | RL loss = -0.02588 | avg reward = 1.1036
Step 390 | RL loss = -0.04485 | avg reward = 1.0611
Step 400 | RL loss = -0.01331 | avg reward = 1.1482


<lambdifygenerated-24323>:2: RuntimeWarning: invalid value encountered in log
  return -x + log((1/2)*sin(x))


Step 410 | RL loss = -0.02069 | avg reward = 0.8706
Step 420 | RL loss = -0.03421 | avg reward = 1.0995


<lambdifygenerated-25203>:2: RuntimeWarning: divide by zero encountered in log
  return -x + log(x*sinh(log(x)))


KeyboardInterrupt: 

In [ ]:
import warnings
import numpy as np
import sympy as sp

TRAIN_X = np.linspace(1.0, 5.0, 20)
TEST_X = np.linspace(1.0, 5.0, 200)

def safe_mse(candidate, reference, x_values):

    try:
        candidate_fn = sp.lambdify(
            x,
            candidate,
            modules="numpy"
        )

        reference_fn = sp.lambdify(
            x,
            reference,
            modules="numpy"
        )

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)

            with np.errstate(
                divide="ignore",
                invalid="ignore",
                over="ignore",
                under="ignore"
            ):
                pred = np.asarray(
                    candidate_fn(x_values),
                    dtype=np.complex128
                )

                target = np.asarray(
                    reference_fn(x_values),
                    dtype=np.complex128
                )

        if not np.all(np.isfinite(pred)):
            return np.inf

        if np.max(np.abs(pred.imag)) > 1e-8:
            return np.inf

        return float(
            np.mean(
                (pred.real - target.real) ** 2
            )
        )

    except Exception:
        return np.inf

In [ ]:
def fit_solution_constants(
    expr,
    reference,
    x_values
):
    """
    Fit up to two symbolic coefficient constants
    appearing in the generated ODE2 solution.

    This prevents us from arbitrarily forcing
    a8=1, a9=0.
    """

    coeffs = sorted(
        [
            s for s in expr.free_symbols
            if str(s).startswith("a")
            and str(s)[1:].isdigit()
        ],
        key=lambda s: str(s)
    )

    coeffs = coeffs[:2]

    if len(coeffs) == 0:
        return expr

    try:

        # ----------------------------------------------------
        # Check whether candidate is affine in its constants.
        # ----------------------------------------------------

        for c1 in coeffs:
            if sp.diff(expr, c1, 2) != 0:
                return expr

        if len(coeffs) == 2:
            if sp.diff(
                sp.diff(expr, coeffs[0]),
                coeffs[1]
            ) != 0:
                return expr

        subs_zero = {
            c: 0
            for c in coeffs
        }

        base_expr = expr.subs(subs_zero)

        basis = []

        for c in coeffs:
            basis.append(
                sp.diff(expr, c).subs(subs_zero)
            )

        base_fn = sp.lambdify(
            x,
            base_expr,
            modules="numpy"
        )

        basis_fns = [
            sp.lambdify(
                x,
                b,
                modules="numpy"
            )
            for b in basis
        ]

        ref_fn = sp.lambdify(
            x,
            reference,
            modules="numpy"
        )

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)

            with np.errstate(
                divide="ignore",
                invalid="ignore",
                over="ignore"
            ):

                y = np.asarray(
                    ref_fn(x_values),
                    dtype=np.float64
                )

                base = np.asarray(
                    base_fn(x_values),
                    dtype=np.float64
                )

                A = np.column_stack([
                    np.asarray(
                        fn(x_values),
                        dtype=np.float64
                    )
                    for fn in basis_fns
                ])

        if (
            not np.all(np.isfinite(A))
            or not np.all(np.isfinite(base))
        ):
            return expr

        target = y - base

        fitted, *_ = np.linalg.lstsq(
            A,
            target,
            rcond=None
        )

        substitutions = {
            coeffs[i]: float(fitted[i])
            for i in range(len(coeffs))
        }

        return sp.simplify(
            expr.subs(substitutions)
        )

    except Exception:
        return expr

In [ ]:
def candidate_logprobs_sum(
    hypotheses,
    src_enc,
    src_len
):
    """
    Differentiable log P(sequence | equation).

    IMPORTANT:
    Returns SUM of token log-probabilities,
    not length-normalized beam scores.
    """

    sequences = []

    for _, hypothesis in hypotheses:

        sequences.append(
            hypothesis[1:].long().cpu()
        )

    x2, len2 = ode2_env.batch_sequences(
        sequences
    )

    x2 = x2.to(device)
    len2 = len2.to(device)

    batch_size = len(sequences)

    src_enc_batch = src_enc.expand(
        batch_size,
        -1,
        -1
    )

    src_len_batch = src_len.expand(
        batch_size
    )

    decoder.eval()

    decoded = decoder(
        "fwd",
        x=x2,
        lengths=len2,
        causal=True,
        src_enc=src_enc_batch,
        src_len=src_len_batch
    )

    logits = decoder.proj(
        decoded[:-1]
    )

    log_probs = torch.log_softmax(
        logits,
        dim=-1
    )

    targets = x2[1:]

    token_log_probs = log_probs.gather(
        -1,
        targets.unsqueeze(-1)
    ).squeeze(-1)

    time = torch.arange(
        token_log_probs.size(0),
        device=device
    )

    mask = (
        time[:, None]
        < (len2 - 1)[None, :]
    )

    token_log_probs = (
        token_log_probs * mask
    )

    # THIS is the important change:
    sequence_log_probs = (
        token_log_probs.sum(dim=0)
    )

    return sequence_log_probs

In [ ]:
def sample_candidates(
    n,
    num_samples=10,
    temperature=0.8
):

    src_enc, src_len = encoded_cache[n]

    src_enc_batch = src_enc.expand(
        num_samples,
        -1,
        -1
    )

    src_len_batch = src_len.expand(
        num_samples
    )

    decoder.eval()

    with torch.no_grad():

        generated, lengths = decoder.generate(
            src_enc_batch,
            src_len_batch,
            max_len=200,
            sample_temperature=temperature
        )

    hypotheses = []

    for i in range(num_samples):

        length = int(
            lengths[i].item()
        )

        # Remove initial <EOS>/<BOS>
        # and final <EOS>.
        tokens = generated[
            1:length - 1,
            i
        ].cpu()

        hypotheses.append(
            (0.0, tokens)
        )

    return hypotheses

In [ ]:
def evaluate_candidate(
    hypothesis,
    n
):

    try:

        raw_expr = hypothesis_to_sympy(
            ode2_env,
            hypothesis
        )

        fitted_expr = fit_solution_constants(
            raw_expr,
            exact_solutions[n],
            TRAIN_X
        )

        mse = safe_mse(
            fitted_expr,
            exact_solutions[n],
            TRAIN_X
        )

        if not np.isfinite(mse):
            return -12.0, np.inf, fitted_expr

        reward = -np.log10(
            mse + 1e-12
        )

        return reward, mse, fitted_expr

    except Exception:

        return -12.0, np.inf, None

In [ ]:
def rl_update_step_v2(
    train_ns=(0, 1),
    num_samples=10,
    temperature=0.8
):

    total_loss = 0.0
    reward_values = []

    for n in train_ns:

        # ---------------------------------------------
        # Sample actions from current policy
        # ---------------------------------------------

        hypotheses = sample_candidates(
            n,
            num_samples=num_samples,
            temperature=temperature
        )

        rewards = []

        for hypothesis in hypotheses:

            reward, mse, expr = evaluate_candidate(
                hypothesis,
                n
            )

            rewards.append(reward)

        rewards = torch.tensor(
            rewards,
            dtype=torch.float32,
            device=device
        )

        # ---------------------------------------------
        # Baseline / advantage
        # ---------------------------------------------

        advantages = (
            rewards - rewards.mean()
        )

        std = advantages.std()

        if std > 1e-6:
            advantages = advantages / std

        advantages = torch.clamp(
            advantages,
            -3.0,
            3.0
        )

        # ---------------------------------------------
        # Differentiable log P(actions)
        # ---------------------------------------------

        src_enc, src_len = encoded_cache[n]

        log_probs = candidate_logprobs_sum(
            hypotheses,
            src_enc,
            src_len
        )

        # ---------------------------------------------
        # REINFORCE
        # ---------------------------------------------

        loss_n = -(
            advantages.detach()
            * log_probs
        ).mean()

        total_loss = (
            total_loss + loss_n
        )

        reward_values.extend(
            rewards.detach()
            .cpu()
            .tolist()
        )

    total_loss /= len(train_ns)

    # ---------------------------------------------
    # Update
    # ---------------------------------------------

    optimizer.zero_grad(
        set_to_none=True
    )

    total_loss.backward()

    grad_norm = clip_grad_norm_(
        decoder.parameters(),
        1.0
    )

    optimizer.step()

    return {
        "loss": total_loss.item(),
        "avg_reward": float(
            np.mean(reward_values)
        ),
        "max_reward": float(
            np.max(reward_values)
        ),
        "grad_norm": float(
            grad_norm.item()
        )
    }

In [ ]:
for p in encoder.parameters():
    p.requires_grad = False

for p in decoder.parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW(
    [
        p
        for p in decoder.parameters()
        if p.requires_grad
    ],
    lr=1e-6,
    weight_decay=1e-4
)

In [ ]:
def evaluate_beam(n):

    results = evaluate_lane_beam(
        n,
        x_values=TEST_X,
        verbose=False
    )

    top1 = results[0]

    best = min(
        results,
        key=lambda r: r["mse"]
    )

    return {
        "top1_mse": top1["mse"],
        "best10_mse": best["mse"],
        "best10_rank": best["rank"],
        "top1_expr": top1["expr"],
        "best_expr": best["expr"]
    }

In [ ]:
before_0 = evaluate_beam(0)
before_1 = evaluate_beam(1)

print("BASELINE")
print("n=0:", before_0)
print("n=1:", before_1)

BASELINE
n=0: {'top1_mse': 0.0812750585283139, 'best10_mse': 0.0812750585283139, 'best10_rank': 1, 'top1_expr': -x + log(2*x + 4 + 2/x), 'best_expr': -x + log(2*x + 4 + 2/x)}
n=1: {'top1_mse': 0.03585994244774494, 'best10_mse': 0.03585994244774494, 'best10_rank': 1, 'top1_expr': (x + 1)*exp(-x), 'best_expr': (x + 1)*exp(-x)}


In [ ]:
NUM_STEPS = 300

history = []

for step in range(1, NUM_STEPS + 1):

    metrics = rl_update_step_v2(
        train_ns=(0, 1),
        num_samples=10,
        temperature=0.8
    )

    if step == 1 or step % 10 == 0:

        print(
            f"Step {step:3d} | "
            f"loss={metrics['loss']:.5f} | "
            f"reward={metrics['avg_reward']:.4f} | "
            f"max={metrics['max_reward']:.4f} | "
            f"grad={metrics['grad_norm']:.4f}"
        )

    # -------------------------------------------------
    # FIXED evaluation every 25 steps
    # -------------------------------------------------

    if step % 25 == 0:

        eval0 = evaluate_beam(0)
        eval1 = evaluate_beam(1)

        history.append({
            "step": step,
            "n0_top1": eval0["top1_mse"],
            "n0_best10": eval0["best10_mse"],
            "n1_top1": eval1["top1_mse"],
            "n1_best10": eval1["best10_mse"],
        })

        print(
            f"\nEVAL @ {step}"
        )

        print(
            f"n=0 | "
            f"top1={eval0['top1_mse']:.6e} | "
            f"best10={eval0['best10_mse']:.6e} | "
            f"rank={eval0['best10_rank']}"
        )

        print(
            f"n=1 | "
            f"top1={eval1['top1_mse']:.6e} | "
            f"best10={eval1['best10_mse']:.6e} | "
            f"rank={eval1['best10_rank']}"
        )

Step   1 | loss=0.00000 | reward=-12.0000 | max=-12.0000 | grad=0.0000
Step  10 | loss=0.00000 | reward=-12.0000 | max=-12.0000 | grad=0.0000
Step  20 | loss=0.00000 | reward=-12.0000 | max=-12.0000 | grad=0.0000

EVAL @ 25
n=0 | top1=8.127506e-02 | best10=8.127506e-02 | rank=1
n=1 | top1=3.585994e-02 | best10=3.585994e-02 | rank=1
Step  30 | loss=0.00000 | reward=-12.0000 | max=-12.0000 | grad=0.0000
Step  40 | loss=0.00000 | reward=-12.0000 | max=-12.0000 | grad=0.0000
Step  50 | loss=0.00000 | reward=-12.0000 | max=-12.0000 | grad=0.0000

EVAL @ 50
n=0 | top1=8.127506e-02 | best10=8.127506e-02 | rank=1
n=1 | top1=3.585994e-02 | best10=3.585994e-02 | rank=1
Step  60 | loss=0.00000 | reward=-12.0000 | max=-12.0000 | grad=0.0000
Step  70 | loss=0.00000 | reward=-12.0000 | max=-12.0000 | grad=0.0000

EVAL @ 75
n=0 | top1=8.127506e-02 | best10=8.127506e-02 | rank=1
n=1 | top1=3.585994e-02 | best10=3.585994e-02 | rank=1
Step  80 | loss=0.00000 | reward=-12.0000 | max=-12.0000 | grad=0.000

KeyboardInterrupt: 

In [ ]:
checkpoint = torch.load(
    '/content/ode2_lane_emden_rl.pth',
    map_location=device
)

encoder.load_state_dict(checkpoint['encoder'])
decoder.load_state_dict(checkpoint['decoder'])

encoder.eval()
decoder.eval()

print("Restored pretrained ODE2 model.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/ode2_lane_emden_rl.pth'

In [ ]:
torch.save(
    {
        'encoder': encoder.state_dict(),
        'decoder': decoder.state_dict(),
    },
    '/content/ode2_lane_emden_rl.pth'
)

print("RL checkpoint saved.")

RL checkpoint saved.


## 10. Evaluating Post-RL Model Performance

In [ ]:
after_n0 = evaluate_lane_beam(
    0,
    x_values=np.linspace(1, 5, 200)
)

after_n1 = evaluate_lane_beam(
    1,
    x_values=np.linspace(1, 5, 200)
)

Lane–Emden n=0
 1 | beam= -0.0497 | MSE=  7.6745e-02 | reward=   1.115
     -x + log(2*(x**2 + Abs(x) + 2)/x)
 2 | beam= -0.0598 | MSE=  7.6745e-02 | reward=   1.115
     -x + log(2*x + 2 + 4/x)
 3 | beam= -0.0632 | MSE=  7.6745e-02 | reward=   1.115
     -x + log(2*(x**2 + Abs(x) + 2)/x)
 4 | beam= -0.0749 | MSE=  7.6745e-02 | reward=   1.115
     -x + log(2*x + 2 + 4/x)
 5 | beam= -0.1538 | MSE=  7.3547e-02 | reward=   1.133
     -x + log((x**2 + sqrt(2*x + 1) + 2)/x) + log(2)
 6 | beam= -0.1665 | MSE=  7.1336e-02 | reward=   1.147
     -x + log((x**2 + sqrt(4*x + 1) + 2)/x) + log(2)
 7 | beam= -0.2429 | MSE=  7.0994e-02 | reward=   1.149
     -x + log((x**2 + sqrt(3*x + 1) + 2)/x) + log(2)
 8 | beam= -0.2598 | MSE=  7.3981e-02 | reward=   1.131
     -x + log(2*x + 2 + 6/x)
 9 | beam= -0.2755 | MSE=  1.6853e-01 | reward=   0.773
     -x + log(2*x + 6 + 4/x)
10 | beam= -0.2758 | MSE=  7.3423e-02 | reward=   1.134
     -x + log((x**2 + sqrt(5*x + 1) + 2)/x) + log(2)
Lane–Emden n=1
 1 |

In [ ]:
def summarize_results(name, results):

    top1 = results[0]

    best = min(
        results,
        key=lambda r: r['mse']
    )

    return {
        'case': name,
        'top1_mse': top1['mse'],
        'top1_reward': top1['reward'],
        'best10_mse': best['mse'],
        'best10_reward': best['reward'],
        'best10_rank': best['rank'],
    }


# Baseline on 200 points
baseline_n0_200 = evaluate_lane_beam(
    0,
    x_values=np.linspace(1, 5, 200),
    verbose=False
)

baseline_n1_200 = evaluate_lane_beam(
    1,
    x_values=np.linspace(1, 5, 200),
    verbose=False
)


before = [
    summarize_results(
        'n=0',
        baseline_n0_200
    ),
    summarize_results(
        'n=1',
        baseline_n1_200
    )
]

after = [
    summarize_results(
        'n=0',
        after_n0
    ),
    summarize_results(
        'n=1',
        after_n1
    )
]


print("\n" + "="*90)
print("BEFORE RL")
print("="*90)

for row in before:
    print(row)


print("\n" + "="*90)
print("AFTER RL")
print("="*90)

for row in after:
    print(row)


BEFORE RL
{'case': 'n=0', 'top1_mse': 0.0767446262459387, 'top1_reward': 1.1149520246195992, 'best10_mse': 0.07099359884800621, 'best10_reward': 1.1487808077581019, 'best10_rank': 7}
{'case': 'n=1', 'top1_mse': 0.03585994244774494, 'top1_reward': 1.4453904117680512, 'best10_mse': 0.03585994244774494, 'best10_reward': 1.4453904117680512, 'best10_rank': 1}

AFTER RL
{'case': 'n=0', 'top1_mse': 0.0767446262459387, 'top1_reward': 1.1149520246195992, 'best10_mse': 0.07099359884800621, 'best10_reward': 1.1487808077581019, 'best10_rank': 7}
{'case': 'n=1', 'top1_mse': 0.03585994244774494, 'top1_reward': 1.4453904117680512, 'best10_mse': 0.03585994244774494, 'best10_reward': 1.4453904117680512, 'best10_rank': 1}


## 11. Debugging Optimizer & Parameter Updates

In [ ]:
# Check that optimizer and decoder refer to the same parameters

decoder_param_ids = {
    id(p) for p in decoder.parameters()
    if p.requires_grad
}

optimizer_param_ids = {
    id(p)
    for group in optimizer.param_groups
    for p in group['params']
}

print("Trainable decoder params:", len(decoder_param_ids))
print("Optimizer params:", len(optimizer_param_ids))

print(
    "Intersection:",
    len(decoder_param_ids & optimizer_param_ids)
)

print(
    "All optimizer params belong to decoder:",
    optimizer_param_ids <= decoder_param_ids
)

Trainable decoder params: 161
Optimizer params: 161
Intersection: 161
All optimizer params belong to decoder: True


In [ ]:
import copy

decoder_before = {
    name: p.detach().clone()
    for name, p in decoder.named_parameters()
    if p.requires_grad
}

In [ ]:
loss, avg_reward, info = rl_update_step(
    train_ns=(0, 1),
    beam_size=10
)

print("RL loss:", loss)
print("Average reward:", avg_reward)

RL loss: -0.00706386286765337
Average reward: 0.8607656076550484


In [ ]:
total_change = 0.0
max_change = 0.0

for name, p in decoder.named_parameters():

    if not p.requires_grad:
        continue

    change = (
        p.detach() - decoder_before[name]
    ).abs()

    total_change += change.sum().item()
    max_change = max(
        max_change,
        change.max().item()
    )

print("Total absolute parameter change:", total_change)
print("Maximum parameter change:", max_change)

Total absolute parameter change: 36.41958895975239
Maximum parameter change: 2.2314488887786865e-06


In [ ]:
total_grad = 0.0
max_grad = 0.0
num_grads = 0

for name, p in decoder.named_parameters():

    if p.grad is None:
        continue

    g = p.grad.detach().abs()

    total_grad += g.sum().item()
    max_grad = max(
        max_grad,
        g.max().item()
    )

    num_grads += 1

print("Parameters with gradients:", num_grads)
print("Total gradient magnitude:", total_grad)
print("Maximum gradient:", max_grad)

Parameters with gradients: 161
Total gradient magnitude: 1827.9908029185335
Maximum gradient: 0.045051973313093185


In [ ]:
for n in [0, 1]:

    results = evaluate_lane_beam(
        n,
        x_values=TRAIN_X,
        verbose=False
    )

    print(f"\nn={n}")

    for r in results:
        print(
            f"rank={r['rank']:2d} "
            f"MSE={r['mse']:.6e} "
            f"reward={r['reward']:.4f}"
        )


n=0
rank= 1 MSE=5.382011e-01 reward=0.2691
rank= 2 MSE=2.480801e-01 reward=0.6054
rank= 3 MSE=2.416419e-01 reward=0.6168
rank= 4 MSE=3.506356e-01 reward=0.4551
rank= 5 MSE=5.382011e-01 reward=0.2691
rank= 6 MSE=2.480801e-01 reward=0.6054
rank= 7 MSE=2.480801e-01 reward=0.6054
rank= 8 MSE=5.382011e-01 reward=0.2691
rank= 9 MSE=2.416419e-01 reward=0.6168
rank=10 MSE=2.416419e-01 reward=0.6168

n=1
rank= 1 MSE=3.564576e-02 reward=1.4480
rank= 2 MSE=3.564576e-02 reward=1.4480
rank= 3 MSE=3.564576e-02 reward=1.4480
rank= 4 MSE=3.564576e-02 reward=1.4480
rank= 5 MSE=4.831898e-02 reward=1.3159
rank= 6 MSE=4.831898e-02 reward=1.3159
rank= 7 MSE=1.284204e-01 reward=0.8914
rank= 8 MSE=8.391308e-02 reward=1.0762
rank= 9 MSE=8.391308e-02 reward=1.0762
rank=10 MSE=1.517557e-01 reward=0.8189
